# Neuron DSI: Signal Infinite Cascade (SIC) Simulation
This script implements a high-throughput simulation pipeline to calculate neuronal responses using the Signal Infinite Cascade (SIC) model. It maps the functional output of sensory circuits under Moving Edge stimuli to determine the Direction Selectivity Index (DSI).

In [ ]:
import sys
import os

# -----------------------------
# DEPENDENCIES
# -----------------------------
sys.path.append(os.path.abspath('../util'))

from stimulus_Loader import StimulusProcessor
from SIC import SICModelTorch

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from datetime import datetime
import csv
import glob
import torch

# ==================================================
# Result directory (npz)
# ==================================================
RESPONSE_ROOT = "../results/responses/dsi"


# ==================================================
# Global npz deduplication (scan ALL response folders)
# ==================================================
print("Scanning existing npz files for global deduplication...")
existing_npz = set()

for npz_file in glob.glob(
    os.path.join(RESPONSE_ROOT, "**", "*.npz"),
    recursive=True
):
    base = os.path.splitext(os.path.basename(npz_file))[0]
    existing_npz.add(base)

print(f"Found {len(existing_npz)} existing response files")


# ---------------------------
# Initialize StimulusProcessor
# ---------------------------
print("\nInitializing StimulusProcessor...")
stimulus_processor = StimulusProcessor(
    target_dt_ms=1.0,
    target_size=(41, 82),
    is_visual=False,
    fps=1000.0
)


# ---------------------------
# Load stimulus videos (AFTER global dedup)
# ---------------------------
video_files = sorted(glob.glob("./stimulus/MovingEdge/*.npz"))

if not video_files:
    raise RuntimeError("❌ No .gif files found.")

print(f"Found {len(video_files)} video files")

stimulus_dataset = {}
skipped = 0

print("\nLoading stimuli (global npz dedup)...")
for v_file in tqdm(video_files, desc="Loading stimuli", unit="video"):

    stim_base = os.path.splitext(os.path.basename(v_file))[0]

    if stim_base in existing_npz:
        skipped += 1
        continue

    stimulus, _ = stimulus_processor.process_npz(
        v_file,
        verbose=False
    )

    key_name = os.path.splitext(v_file)[0]
    stimulus_dataset[key_name] = stimulus


print(f"✅ Stimuli loaded: {len(stimulus_dataset)}")
print(f"⏭️  Stimuli skipped (existing npz anywhere): {skipped}")


# ==================================================
# Initialize LMCs model (AFTER stimulus filtering)
# ==================================================
print("\nInitializing LMCs model...")
SIC_model = SICModelTorch(
    matrix_dir="neuron_matrices",
    output_dir="../results/responses/fri",
    t_step=10,
    rate=100,
    device=torch.device("cuda:0")
)


# ---------------------------
# Load neuron IDs
# ---------------------------
def read_all_neuron_ids(filename='./data/classification.txt'):
    neuron_ids = []
    try:
        with open(filename, 'r') as f:
            reader = csv.DictReader(f)
            for row in reader:
                if 'root_id' in row and row['root_id'].strip():
                    try:
                        neuron_ids.append(int(row['root_id'].strip()))
                    except ValueError:
                        continue
    except FileNotFoundError:
        print(f"Error: File '{filename}' not found")
    except Exception as e:
        print(f"Error reading file: {str(e)}")
    return neuron_ids


print("\nLoading neuron weights...")
neuron_ids = read_all_neuron_ids()
print(f"Found {len(neuron_ids)} neurons")

weights = SIC_model.load_weights(neuron_ids,normalize=True) ## normalize weights to [-1,1]
print("Weights loaded successfully")


# ---------------------------
# Calculate responses
# ---------------------------
print("\nCalculating LMC responses...")
for stim_name, stimulus in tqdm(
    stimulus_dataset.items(),
    desc="Processing responses",
    unit="stimulus"
):
    name = os.path.splitext(os.path.basename(stim_name))[0]

    SIC_model.calculate_response_baseline(
        stimulus,
        weights,
        responce_threshold=0,
        baseline_steps=50,
        stim_name=name
    )


# ---------------------------
# Summary
# ---------------------------
print("\n" + "=" * 50)
print("PREPROCESSING COMPLETE")
print("=" * 50)
print(f"Total video files found: {len(video_files)}")
print(f"Stimuli skipped (existing npz): {skipped}")
print(f"Stimuli processed: {len(stimulus_dataset)}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 50)


Generating two custom stimuli...
✅ Total stimuli: 2
Using device: cuda:1
Found 139255 neurons
Weights loaded successfully
Calculating SIC responses...


Processing stimuli: 100%|██████████| 2/2 [00:23<00:00, 11.69s/stimulus]


PREPROCESSING COMPLETE
Total stimuli processed: 2
Timestamp: 2026-03-10 17:13:23
